# Analyzing SAE — feature dashboards + per-token probes

End-to-end exploration of one trained SAE: feature dashboards (precomputed by `saes/runInference.py`), per-feature DLA, ablations, steering, and the original per-token analyses that the new `featureBucketing.ipynb` generalizes.

### Syncing precomputed inference from PSC

```bash
rsync -av friedmae@bridges2:Interp_LM4/saes/{sae_runs,sae_inference}/ \
    ~/Code/Project\ Code/CRL-Interp/Interp_LM4/saes/
```

After that, `SAE_PATH` below can point at any `saes/sae_runs/<model>/.../final` and the matching `saes/sae_inference/<model>/<trial>/` is found automatically.

In [1]:
import sys
from pathlib import Path

# Put the repo root on sys.path so `util.*` and `saes.*` resolve regardless of
# whether the kernel was launched from `Interp_LM4/` or `saes/`.
REPO_ROOT = Path.cwd() if (Path.cwd() / "saes").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# List every trained SAE under saes/sae_runs/. Copy one of these paths into
# SAE_PATH in the next cell.
from pathlib import Path

SAE_RUNS_DIR = REPO_ROOT / "saes" / "sae_runs"
finals = sorted(SAE_RUNS_DIR.rglob("final")) if SAE_RUNS_DIR.exists() else []
print(f"{len(finals)} trained SAE(s) under {SAE_RUNS_DIR.relative_to(REPO_ROOT)}:\n")
for p in finals:
    print(f"  {p.relative_to(REPO_ROOT)}")

In [ ]:
from pathlib import Path

MODEL_DIR  = Path("../model/BD_llama_6heads_1epoch_4layers")
DATA_DIR   = Path("../data/bioS_N-Bd_final_grid")
REMAP_PATH = DATA_DIR / "old_to_new.json"
TOKENS_PATH = DATA_DIR / "bios_postreduce.bin"

# Pick one SAE. Run the discovery cell below to list available paths.
# Layout produced by saes/trainSAE.py + saes/runInference.py:
#   saes/sae_runs/<model_name>/<sweep_folder>/<trial_name>/final
#   saes/sae_inference/<model_name>/<trial_name>/{feature_stats,buckets,dashboards}
SAE_PATH = REPO_ROOT / "saes" / "sae_runs" / "grid-L8-H6" / "sweep-XXXXXXXX" / "L2_mult8_l02_lr3e-05_ep50_n10000" / "final"

# Derive model + trial names so the matching inference dir is found
# automatically. parents[2] skips the <sweep_folder> segment.
TRIAL_NAME = SAE_PATH.parent.name
MODEL_NAME = SAE_PATH.parents[2].name
INFERENCE_DIR = REPO_ROOT / "saes" / "sae_inference" / MODEL_NAME / TRIAL_NAME

print(f"SAE_PATH:      {SAE_PATH.relative_to(REPO_ROOT)}  (exists: {SAE_PATH.exists()})")
print(f"MODEL_NAME:    {MODEL_NAME}")
print(f"TRIAL_NAME:    {TRIAL_NAME}")
print(f"INFERENCE_DIR: {INFERENCE_DIR.relative_to(REPO_ROOT)}  (exists: {INFERENCE_DIR.exists()})")

### Load Model, Data, and Tokenizer

In [3]:
from util.condensed_tokenizer import CondensedTokenizer
from util.bio_sampler import BioSampler

tokenizer = CondensedTokenizer.from_remap_path(REMAP_PATH)
sampler   = BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=0)

print(f"vocab_size = {tokenizer.vocab_size}, eos_token_id = {tokenizer.eos_token_id}")
print(f"{len(sampler.people):,} people, {sampler.n_templates} templates/person\n")

# Specific person + specific template
text = sampler.render(sampler.people[0], exposure_idx=0)
print("render(people[0], 0) →", repr(text))
print("encode →", tokenizer.encode(text)[:15], "...\n")

# Random bio
draw = sampler.sample()
print(f"sample() → person id={draw['person']['id']}, template={draw['exposure_idx']}")
print("text →", repr(draw["text"]))

/Users/efmac/Code/Project Code/CRL-Interp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab_size = 1836, eos_token_id = 1835
50,000 people, 46 templates/person

render(people[0], 0) → ' Gabriella Ella Rigby was born on February 18, 1816.'
encode → [870, 83, 882, 663, 5, 1273, 267, 80, 536, 52, 487, 237, 1, 237, 256] ...

sample() → person id=50494, template=26
text → ' Marco Jackson Rowland arrived in this world on December 24, 1717, a day to be remembered.'


In [ ]:
def find_person(full_name):
    """Look up a person by name. Tries 'First Middle Last', then 'First Last'.
    Returns the matching dict from sampler.people, or None if no match."""
    target = full_name.strip()
    for p in sampler.people:
        if f"{p['first_name']} {p['middle_name']} {p['last_name']}" == target:
            return p
        if f"{p['first_name']} {p['last_name']}" == target:
            return p
    return None


In [4]:
import torch
from transformers import LlamaForCausalLM
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_llama_weights
from sae_lens import HookedSAETransformer

def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = pick_device()
dtype = torch.float32

hf_model = LlamaForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=dtype)
hf_model.eval()
assert hf_model.config.vocab_size == tokenizer.vocab_size, (
    f"checkpoint vocab {hf_model.config.vocab_size} != remap vocab "
    f"{tokenizer.vocab_size} — wrong old_to_new.json for this model."
)

# Build the TL config from the HF config so dims match our custom
# 4-layer / hidden=384 / vocab=1836 model (from_pretrained would have used
# the Llama-2-7b template config and tried to read layer 4 of a 4-layer model).
hf_cfg = hf_model.config
tl_cfg = HookedTransformerConfig(
    n_layers = hf_cfg.num_hidden_layers,
    d_model = hf_cfg.hidden_size,
    d_head = hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    n_heads = hf_cfg.num_attention_heads,
    d_mlp= hf_cfg.intermediate_size,
    d_vocab=hf_cfg.vocab_size,
    n_ctx=hf_cfg.max_position_embeddings,
    act_fn="silu",
    normalization_type="RMS",
    gated_mlp=True,
    positional_embedding_type="rotary",
    rotary_base=int(getattr(hf_cfg, "rope_theta", 10000.0)),
    rotary_dim=hf_cfg.hidden_size // hf_cfg.num_attention_heads,
    final_rms=True,
    tie_word_embeddings=hf_cfg.tie_word_embeddings,
    initializer_range=hf_cfg.initializer_range,
    n_key_value_heads=hf_cfg.num_key_value_heads,
    device=device,
)

# Pre-tokenize everything you feed the model; don't attach the tokenizer.
# TL only calls into the tokenizer for `model(str)` / `to_tokens` / `to_string`,
# none of which we use — we always pass token ids directly.
state_dict = convert_llama_weights(hf_model, tl_cfg)
model = HookedSAETransformer(tl_cfg)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print(f"Loaded on {device}: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}, "
      f"n_heads={model.cfg.n_heads}, d_vocab={model.cfg.d_vocab}")

`torch_dtype` is deprecated! Use `dtype` instead!


Moving model to device:  mps
Loaded on mps: n_layers=4, d_model=384, n_heads=6, d_vocab=1836


In [5]:
def show_tokens(ids, tokenizer, addOne=False):
    """Print each token with its position and decoded text (with repr so
    leading spaces / newlines stay visible).

    addOne=True shifts the displayed index by 1, so positions match what
    the model sees after a BOS/EOS is prepended to `ids` downstream.
    """
    if hasattr(ids, "tolist"):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]   # unwrap [1, N] batch

    offset = 1 if addOne else 0
    id_w = max(len(str(t)) for t in ids)
    idx_w = len(str(len(ids) - 1 + offset))
    print(f"{'idx':>{idx_w}} | {'id':>{id_w}} | text")
    print("-" * (idx_w + id_w + 12))
    for i, t in enumerate(ids):
        print(f"{i + offset:>{idx_w}} | {t:>{id_w}} | {tokenizer.decode([t])!r}")


### Exploring SAE 

#### Setup: load the SAE

The SAE checkpoint is loaded once and reused by every subsequent cell.

In [ ]:
from saes.evalSAE import load_sae

sae = load_sae(SAE_PATH, device)
HOOK = sae.cfg.hook_name   # read from saved cfg — auto-tracks the layer
print(f"d_sae = {sae.cfg.d_sae}, hook = {HOOK}")

#### Overview — feature activity histograms

Loads `feature_stats.pt` written by `saes/runInference.py`. Plots:
- **Activation density** per feature (fraction of tokens where the feature fires).
  The big spike at 0 is dead features.
- **Max activation** per feature.

If the file isn't there yet, run `saes/runInference.py` on the HPC and rsync `saes/sae_inference/` over (see the rsync snippet at the top of this notebook).

In [ ]:
import torch
import matplotlib.pyplot as plt

STATS_PATH = INFERENCE_DIR / "feature_stats.pt"
print("stats: ", STATS_PATH, " exists:", STATS_PATH.exists())

if STATS_PATH.exists():
    stats = torch.load(STATS_PATH, weights_only=False)
    n_dead = int((stats["activation_count"] == 0).sum())
    print(f"d_sae = {sae.cfg.d_sae},  n_tokens = {stats['n_tokens']:,},  dead = {n_dead}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    density = stats["activation_count"].float() / stats["n_tokens"]
    axes[0].hist(density.numpy(), bins=50, log=True)
    axes[0].set_xlabel("activation density (fraction of tokens)")
    axes[0].set_ylabel("# features (log)")
    axes[0].set_title(f"activation density — {n_dead} dead / {sae.cfg.d_sae}")

    axes[1].hist(stats["max_activation"].numpy(), bins=50, log=True)
    axes[1].set_xlabel("max activation across corpus")
    axes[1].set_ylabel("# features (log)")
    axes[1].set_title("max activation")
    plt.tight_layout()
    plt.show()
else:
    print()
    print("⚠ feature_stats.pt missing. Run saes/runInference.py on the HPC, then rsync:")
    print("   rsync -av friedmae@bridges2:Interp_LM4/saes/{sae_runs,sae_inference}/ <local-repo>/saes/")

#### Activations per latent

Bar plot of how many tokens each feature fires on, indexed by latent.
Tall bars are densely-firing features; bars at zero are dead. Same data
that drives the activation-density histogram above, plotted along the
latent axis instead of binned.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

TOP_N = 10  # label the N most-active latents

STATS_PATH = INFERENCE_DIR / "feature_stats.pt"

if STATS_PATH.exists():
    stats = torch.load(STATS_PATH, weights_only=False)
    counts = stats["activation_count"].numpy()
    n_total = stats["n_tokens"]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(counts)), counts, width=1.0, linewidth=0)
    ax.set_xlabel("latent index")
    ax.set_ylabel(f"# activations (out of {n_total:,} tokens)")
    ax.set_title(f"Activations per latent  ({sae.cfg.d_sae} features, {int((counts == 0).sum())} dead)")
    ax.set_xlim(0, len(counts))

    top_idx = np.argsort(counts)[-TOP_N:][::-1]
    for idx in top_idx:
        ax.annotate(
            str(idx),
            xy=(idx, counts[idx]),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center", va="bottom",
            fontsize=8,
        )
    # Headroom so labels aren't clipped
    ax.set_ylim(0, counts.max() * 1.08)

    plt.tight_layout()
    plt.show()

    print(f"\nTop {TOP_N} latents by activation count:")
    print(f"{'rank':>4}  {'latent':>6}  {'count':>10}  {'density':>8}")
    for rank, idx in enumerate(top_idx, 1):
        print(f"{rank:>4}  {idx:>6}  {counts[idx]:>10,}  {counts[idx]/n_total:>8.3%}")
else:
    print("⚠ feature_stats.pt missing. Run saes/runInference.py on the HPC, then rsync over.")

#### Option A — View precomputed dashboard&nbsp;&nbsp; ← RECOMMENDED

Run `saes/runInference.py` on the HPC, then rsync (see top of notebook).

Change the index passed to `show_feature` below to inspect a different feature. Each per-feature HTML is ~500 KB, small enough to embed inline. To see a list of all rendered features, open `index.html` inside `INFERENCE_DIR` in your browser.

In [ ]:
import html as _html
import re
from IPython.display import display, HTML

def show_feature(feature_idx, width="100%", height=900):
    """Render the precomputed per-feature dashboard inline in the notebook.

    width  : CSS string ("100%", "800px") or int (px). Default "100%".
    height : CSS string or int (px). Default 900.
    """
    feature_html = INFERENCE_DIR / f"dashboard_feature_{feature_idx}.html"
    if not feature_html.exists():
        if not INFERENCE_DIR.exists():
            print(f"⚠ No inference dir at {INFERENCE_DIR}")
            print("   Run saes/runInference.py on the HPC, then rsync over.")
            return
        pat = re.compile(r"dashboard_feature_(\d+)\.html$")
        avail = sorted({int(m.group(1)) for p in INFERENCE_DIR.glob("dashboard_feature_*.html")
                                       for m in [pat.search(p.name)] if m})
        print(f"⚠ Feature {feature_idx} has no panel. {len(avail)} rendered features.")
        if avail:
            print(f"   First few: {avail[:10]}")
        return

    w = f"{width}px" if isinstance(width, int) else str(width)
    h = f"{height}px" if isinstance(height, int) else str(height)
    srcdoc = _html.escape(feature_html.read_text(), quote=True)
    display(HTML(
        f'<iframe srcdoc="{srcdoc}" width="{w}" height="{h}" '
        f'style="border:1px solid #ccc"></iframe>'
    ))


# Example — change the index, or call show_feature(idx, height=1200) etc.
show_feature(6010)

#### Option B — Compute the dashboard locally

Use this only on a GPU machine or when you can wait 10–30+ minutes on MPS.
It runs the model + SAE forward over the full bio corpus and writes the
same `dashboard.html` that Option A reads. For fast iteration, pass
`features=range(100)` to `make_dashboard` to render a feature subset.

In [ ]:
from saes.sae_explorer import build_index_corpus

# Reuse the inference dir's index_corpus.pt so feature numbers line up
# with the precomputed dashboards (Option A).
tokens = build_index_corpus(
    sampler,
    tokenizer,
    n_per_person=2,
    context_size=64,
    seed=0,
    cache_path=INFERENCE_DIR / "index_corpus.pt",
)
print("corpus shape:", tokens.shape)  # expect [N, 64]

In [ ]:
from saes.sae_explorer import make_dashboard

# Full pass: every feature in the SAE over the index corpus.
# Expect 10–30 min on MPS, much faster on CUDA.
# Pass features=range(100) (or similar) to render a subset while iterating.
out_html = make_dashboard(
    model,
    sae,
    tokens.to(device),
    tokenizer,
    out_dir=INFERENCE_DIR,
    hook_name=HOOK,
)
print("open in browser:", out_html)

#### Breakingdown a Features on a Prompt

Two ways to choose a `feature_idx` for the DLA and steering cells below:

1. **Visual** — scroll `index.html` from Option A, click panels until you find
   one with distinctive top activating examples.
2. **Programmatic** — feed a sample bio to the cell below; it returns the
   features that fire hardest on that input. Copy a `feature_idx` from the
   table into the DLA / steer cells.

Verifying a prompt with features
1. Sample a person
2. Take the prompt up to the persons BD, feed it into the model
3. if the model gets it right(it should), make the SAE reactivate it
3. If the model with sae gets it correct, feed the prompt into the SAE for hgihest activations


In [ ]:
eos    = tokenizer.eos_token_id               
HOOK   = "blocks.1.hook_mlp_out"
device = next(model.parameters()).device      # whatever the model is on
sae    = sae.to(device).eval()

@torch.no_grad()
def greedy(prefix_ids, max_new_tokens=32):
    """Greedy free-decode from prefix_ids, stop at EOS. Returns reduced-vocab ids.
    Pass fwd_hooks to splice something in (e.g. the SAE) during each forward."""
    cur, generated = torch.tensor([prefix_ids], device=device), []
    for _ in range(max_new_tokens):
        logits = model(cur, return_type="logits")
        nxt = int(logits[0, -1].argmax())
        if nxt == eos:                         # "until EOS is hit"
            break
        generated.append(nxt)
        cur = torch.cat([cur, torch.tensor([[nxt]], device=device)], dim=1)
    return generated

@torch.no_grad()
def greedy_w_sae(prefix_ids,sae, max_new_tokens=32):
    """Greedy free-decode from prefix_ids, stop at EOS. Returns reduced-vocab ids.
    Pass fwd_hooks to splice something in (e.g. the SAE) during each forward."""
    cur, generated = torch.tensor([prefix_ids], device=device), []
    for _ in range(max_new_tokens):
        logits = model.run_with_saes(cur, saes=[sae], return_type="logits")
        nxt = int(logits[0, -1].argmax()) #choose last token and
        if nxt == eos:                         # "until EOS is hit"
            break
        generated.append(nxt)
        cur = torch.cat([cur, torch.tensor([[nxt]], device=device)], dim=1)
    return generated


In [ ]:
sample = sampler.sample()
text = sample['text']
print(sample)
birthString = f"{sample['person']["birthmonth"]} {sample['person']['birthday']}, {sample['person']['birthyear']}."
if birthString in text:
    prompt = text.replace(birthString, "").rstrip()
    print(f"Replaced bio with prompt \n Bio: {text} \n Prompt: {prompt}")
else:
    print("Manually replace it")
    
    

{'person': {'id': 53075, 'first_name': 'Natalia', 'middle_name': 'Camila', 'last_name': 'Hodgkinson', 'birthday': 25, 'birthmonth': 'December', 'birthyear': 1826, 'birthcity': 'Salinas, CA', 'university': 'University of Miami', 'field': 'Computer Science', 'company1name': 'Amazon', 'company1city': 'Seattle, WA'}, 'exposure_idx': 19, 'text': ' Natalia Camila Hodgkinson came into existence on the significant date of December 25, 1826.'}
Replaced bio with prompt 
 Bio:  Natalia Camila Hodgkinson came into existence on the significant date of December 25, 1826. 
 Prompt:  Natalia Camila Hodgkinson came into existence on the significant date of


In [ ]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(prompt.rstrip()) #add eos token. 
normalModelIDs = greedy(ids)
modelReturn = tokenizer.decode(normalModelIDs)

saeModelIDs = greedy_w_sae(ids, )

' December 25, 1826.'

In [60]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device).squeeze()
show_tokens(input_tokens, tokenizer)

idx |   id | text
------------------
 0 | 1835 | ''
 1 | 1834 | ' Giovanni'
 2 | 1771 | ' Weston'
 3 | 1694 | ' Greenwood'
 4 | 1377 | ' commemor'
 5 |  153 | 'ates'
 6 |  118 | ' their'
 7 |  495 | ' birth'
 8 |  826 | ' anniversary'
 9 |   52 | ' on'
10 |  426 | ' October'
11 |  242 | ' 15'
12 |    1 | ','
13 |  237 | ' 18'
14 |  241 | '15'
15 |    2 | '.'


In [41]:
# Does model get it correctly?
tokenizer.decode(model.generate(input_tokens[:promptEnd]))

IndexError: tuple index out of range

In [ ]:
from saes.sae_explorer import top_features_for_text

text = sample['text']
print(f"input: {text!r}\n")

rows = top_features_for_text(model, sae, tokenizer, text, HOOK, k=15)
print(f"{'idx':>6} {'max':>8} {'mean':>8} {'@pos':>5}  token")
print("-" * 50)
for r in rows:
    print(f"  {r['feature_idx']:>4}  {r['max_activation']:7.3f}  "
          f"{r['mean_activation']:7.3f}  {r['position_argmax']:>4}   "
          f"{r['token_at_argmax']!r}")

input: ' Veronica Autumn Ashworth acknowledges their birth day as November 6, 1794.'

   idx      max     mean  @pos  token
--------------------------------------------------
  4153    4.481    0.280     0   ''
  4030    3.475    0.217     0   ''
    43    3.429    0.214     0   ''
   916    3.280    0.205     0   ''
  2229    3.129    0.196     0   ''
   796    3.112    0.325     0   ''
  3110    3.080    0.192     0   ''
   327    2.863    0.179     0   ''
  1895    2.845    0.328     2   ' Autumn'
   910    2.797    0.326     0   ''
  2890    2.733    0.222     0   ''
  2126    2.717    0.170     0   ''
  4324    2.711    0.169     0   ''
  5463    2.685    0.168     0   ''
  2242    2.685    0.168     0   ''


#### Direct Logit Attribution (DLA)

Project a feature's decoder direction through the model's unembed matrix
to see which output tokens it linearly promotes (top) and suppresses
(bottom). The dashboard shows this per-feature; this cell lets you
inspect features programmatically.

Note: DLA ignores downstream attention/MLP and any layernorm scaling.
It's a cheap intuition, not a full causal account — pair with `steer()`
below if you want the real effect on predictions.

In [ ]:
from saes.sae_explorer import dla

# Replace feature_idx with one you found interesting in the dashboard.
result = dla(sae, model, tokenizer, feature_idx=0, k=10)
print("=== TOP (promoted) ===")
for r in result["top"]:
    print(f"  {r['logit_delta']:+.3f}  {r['text']!r:>20}  (id={r['token_id']})")
print("\n=== BOTTOM (suppressed) ===")
for r in result["bottom"]:
    print(f"  {r['logit_delta']:+.3f}  {r['text']!r:>20}  (id={r['token_id']})")

#### Causal probes — `steer()`

Boost one feature's decoder direction at the hook, compare next-token
predictions with and without the boost. Fast even on a laptop (one
forward pass on a short input).

In [ ]:
from saes.sae_explorer import steer

# Pick a feature from the dashboard's dropdown and probe it causally.
result = steer(
    model, sae, tokenizer,
    text=" Gabriella Ella Rigby was born on",
    feature_idx=0,         # replace with a feature that looked interesting
    scale=5.0,
    hook_name=HOOK,
)
for k, rows in result.items():
    print(f"\n=== {k} ===")
    for r in rows:
        print(f"  {r['logit']:+.2f}  {r['text']!r}  (id={r['token_id']})")

### Features for a Token Class

Define a set of input tokens you care about (months, years, names, ...) and find the SAE features that consistently take up the biggest share of activation mass when those tokens appear, plus a specificity score comparing the target-token feature distribution against the baseline of every other (non-pad) token.

Method:
1. For every position in the cached bio corpus, encode SAE features and **L1-normalize** so the `[d_sae]` vector at that position becomes a distribution over features.
2. Average those distributions separately over **target positions** (input token in `TARGET_TOKEN_IDS`) and **other positions** (everything else, minus pad).
3. Rank features by `mean_target` (mass on target) or `specificity = mean_target / mean_other` (selectivity).

Edit `TARGET_STRINGS` in the next cell to retarget; the rest is reusable as-is.


In [14]:
# === Edit this list to retarget. Everything below it is reusable. ===
TARGET_STRINGS = [
    "January", "February", "March",     "April",   "May",      "June",
    "July",    "August",   "September", "October", "November", "December",
]


def resolve_target_token_ids(tokenizer, strings):
    """Encode each string and resolve to a single token id. Tries the
    leading-space form first (' January'), which is what appears mid-sentence
    in the bios; falls back to the bare form. Reports multi-token strings and
    strings whose pieces aren't in the reduced vocab at all."""
    def try_encode(s):
        try:
            return tokenizer.encode(s)
        except KeyError:
            return None  # contains a GPT-2 piece not in the reduced vocab

    target_ids = set()
    rows = []
    for s in strings:
        leading = try_encode(" " + s)
        bare    = try_encode(s)
        if leading is not None and len(leading) == 1:
            target_ids.add(leading[0])
            rows.append((s, "ok", leading[0]))
        elif bare is not None and len(bare) == 1:
            target_ids.add(bare[0])
            rows.append((s, "ok (bare)", bare[0]))
        elif leading is not None:
            target_ids.update(leading)
            rows.append((s, "multi-token", leading))
        elif bare is not None:
            target_ids.update(bare)
            rows.append((s, "multi-token (bare)", bare))
        else:
            rows.append((s, "out-of-vocab", None))
    return target_ids, rows


TARGET_TOKEN_IDS, _report = resolve_target_token_ids(tokenizer, TARGET_STRINGS)

print(f"{'string':>12}  {'status':<19}  id(s)")
print("-" * 48)
for s, status, tid in _report:
    print(f"{s:>12}  {status:<19}  {tid}")
print(f"\n{len(TARGET_TOKEN_IDS)} target token id(s) total")


      string  status               id(s)
------------------------------------------------
     January  ok                   427
    February  ok                   487
       March  ok                   384
       April  ok                   404
         May  ok                   292
        June  ok                   383
        July  ok                   391
      August  ok                   395
   September  ok                   373
     October  ok                   426
    November  ok                   441
    December  ok                   444

12 target token id(s) total


In [ ]:
from saes.sae_explorer import build_index_corpus, features_for_token_set

# Reuse the dashboard's corpus so feature indices line up with the
# precomputed per-feature HTML panels (Option A).
tokens = build_index_corpus(
    sampler,
    tokenizer,
    n_per_person=2,
    context_size=64,
    seed=0,
    cache_path=INFERENCE_DIR / "index_corpus.pt",
)
print(f"corpus: {tuple(tokens.shape)}  ({tokens.shape[0] * tokens.shape[1]:,} positions)")

stats = features_for_token_set(
    model, sae,
    tokens=tokens,
    target_token_ids=TARGET_TOKEN_IDS,
    hook_name=HOOK,
    batch_size=8,
    ignore_token_ids={tokenizer.pad_token_id},
)
print(f"target positions: {stats['n_target_positions']:,}")
print(f"other  positions: {stats['n_other_positions']:,}")

In [9]:
import torch

K = 15

def show_top(stats, sort_key, k=K):
    order = torch.argsort(stats[sort_key], descending=True)[:k]
    print(f"\nTop {k} by {sort_key}")
    print(f"{'rank':>4}  {'idx':>6}  {'mean_target':>11}  {'mean_other':>10}  {'specificity':>11}")
    print("-" * 56)
    for rank, i in enumerate(order, 1):
        i = int(i)
        print(f"{rank:>4}  {i:>6}  "
              f"{float(stats['mean_target'][i]):>11.4f}  "
              f"{float(stats['mean_other'][i]):>10.4f}  "
              f"{float(stats['specificity'][i]):>11.2f}")

show_top(stats, "mean_target")
show_top(stats, "specificity")

# To inspect a candidate: open sae_inference/.../dashboard_feature_<idx>.html,
# or call dla(sae, model, tokenizer, feature_idx=<idx>) for a quick read on
# which output tokens the feature linearly promotes.



Top 15 by mean_target
rank     idx  mean_target  mean_other  specificity
--------------------------------------------------------
   1    1507       0.0086      0.0164         0.52
   2    2336       0.0083      0.0187         0.44
   3    1502       0.0081      0.0075         1.09
   4    5620       0.0079      0.0232         0.34
   5     576       0.0075      0.0076         0.98
   6     991       0.0073      0.0121         0.60
   7     789       0.0072      0.0214         0.34
   8    2473       0.0072      0.0048         1.50
   9    1949       0.0071      0.0168         0.42
  10    4848       0.0070      0.0104         0.67
  11    5266       0.0068      0.0059         1.14
  12    1115       0.0066      0.0079         0.84
  13    5465       0.0066      0.0066         1.00
  14    5663       0.0065      0.0096         0.68
  15    2086       0.0060      0.0029         2.06

Top 15 by specificity
rank     idx  mean_target  mean_other  specificity
------------------------------

### Per-month breakdown — top 5 features per month

Single forward pass over the corpus, but bucketize positions by the *specific*
month token. Shared `"other"` baseline = any non-month, non-pad position.
Filters specificity rankings by a `MIN_TARGET` floor so dead features
don't dominate.


In [15]:
import torch

def _resolve_one(tokenizer, s):
    """Single token id for the leading-space form (preferred) or bare form."""
    for form in (" " + s, s):
        try:
            ids = tokenizer.encode(form)
            if len(ids) == 1:
                return ids[0]
        except KeyError:
            pass
    return None


MONTH_STRINGS = [
    "January", "February", "March",     "April",   "May",      "June",
    "July",    "August",   "September", "October", "November", "December",
]
month_to_id = {m: _resolve_one(tokenizer, m) for m in MONTH_STRINGS}
print("month -> token id:")
for m, tid in month_to_id.items():
    print(f"  {m:>10}: {tid}")

device = next(model.parameters()).device
d_sae = sae.cfg.d_sae
eps = 1e-8

all_month_ids = torch.tensor(
    sorted({i for i in month_to_id.values() if i is not None}),
    device=device, dtype=torch.long,
)
ignore_ids = torch.tensor([tokenizer.pad_token_id], device=device, dtype=torch.long)

per_month_sum = {m: torch.zeros(d_sae, device=device)
                 for m, tid in month_to_id.items() if tid is not None}
per_month_count = {m: 0 for m, tid in month_to_id.items() if tid is not None}
other_sum = torch.zeros(d_sae, device=device)
other_count = 0

BATCH = 32  # tiny model — MPS handles this fine
with torch.no_grad():
    for i in range(0, len(tokens), BATCH):
        batch = tokens[i:i + BATCH].to(device)
        _, cache = model.run_with_cache(batch, names_filter=HOOK)
        feats = sae.encode(cache[HOOK])              # [B, T, d_sae]
        flat_feats = feats.reshape(-1, d_sae)
        flat_tok = batch.reshape(-1)

        norms = flat_feats.sum(dim=1)
        valid = norms > 0
        normed = torch.zeros_like(flat_feats)
        normed[valid] = flat_feats[valid] / norms[valid].unsqueeze(1)

        in_ignore = torch.isin(flat_tok, ignore_ids)
        in_any_month = torch.isin(flat_tok, all_month_ids)

        for m, tid in month_to_id.items():
            if tid is None:
                continue
            mask = (flat_tok == tid) & valid & ~in_ignore
            per_month_sum[m] += normed[mask].sum(dim=0)
            per_month_count[m] += int(mask.sum())

        other_mask = ~in_any_month & valid & ~in_ignore
        other_sum += normed[other_mask].sum(dim=0)
        other_count += int(other_mask.sum())

mean_other = (other_sum / max(other_count, 1)).cpu()
print(f"\nshared 'other' positions: {other_count:,}")

# Floor for specificity ranking. 0.005 = 5x roughly random share of mass.
# Lower it if real features are getting filtered; raise if noise still leaks in.
MIN_TARGET = 0.005
TOP_K = 5

for m in MONTH_STRINGS:
    tid = month_to_id[m]
    if tid is None:
        print(f"\n=== {m}: unresolved ===")
        continue
    n = per_month_count[m]
    if n == 0:
        print(f"\n=== {m}: no target positions ===")
        continue
    mean_target = (per_month_sum[m] / n).cpu()
    specificity = mean_target / (mean_other + eps)
    scores = specificity.clone()
    scores[mean_target < MIN_TARGET] = -float("inf")
    top = torch.argsort(scores, descending=True)[:TOP_K]
    print(f"\n=== {m}  (id={tid}, n={n:,}) ===")
    print(f"{'rank':>4}  {'idx':>6}  {'mean_target':>11}  {'mean_other':>10}  {'specificity':>11}")
    for rank, idx in enumerate(top, 1):
        idx = int(idx)
        print(f"{rank:>4}  {idx:>6}  "
              f"{float(mean_target[idx]):>11.4f}  "
              f"{float(mean_other[idx]):>10.4f}  "
              f"{float(specificity[idx]):>11.2f}")


month -> token id:
     January: 427
    February: 487
       March: 384
       April: 404
         May: 292
        June: 383
        July: 391
      August: 395
   September: 373
     October: 426
    November: 441
    December: 444

shared 'other' positions: 1,329,302

=== January  (id=427, n=8,536) ===
rank     idx  mean_target  mean_other  specificity
   1    5181       0.0054      0.0009         6.25
   2    3689       0.0054      0.0009         5.79
   3    5995       0.0064      0.0016         3.97
   4    3498       0.0055      0.0019         2.95
   5     376       0.0092      0.0032         2.86

=== February  (id=487, n=8,232) ===
rank     idx  mean_target  mean_other  specificity
   1    5181       0.0085      0.0009         9.81
   2    5995       0.0081      0.0016         5.02
   3    1062       0.0062      0.0014         4.44
   4    5407       0.0096      0.0033         2.94
   5    5884       0.0051      0.0018         2.85

=== March  (id=384, n=8,663) ===
rank     

### Exploring SAE - Specific Example

In [ ]:
sample = sampler.sample()

ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device)

show_tokens(tokenizer.encode(sample["text"]), tokenizer, addOne=True)

In [ ]:
ids = [tokenizer.eos_token_id] + tokenizer.encode(sample["text"]) #Add a leading EOS so the first content token has a fresh attention context,
input_tokens = torch.tensor([ids], device=device)
show_tokens(tokenizer.encode(sample["text"]), tokenizer, addOne=True)